# Private odds DataFrame quickstart

[Open in Colab](https://colab.research.google.com/github/JacobiusMakes/parlayapi-notebooks/blob/main/01-quickstart.ipynb)

Inspect nested event data, flatten it into a DataFrame, and optionally export a private CSV.
CSV export is off unless SAVE_PRIVATE_CSV is enabled. No observations are fetched by default.

**Run All is offline by default:** no API calls or key prompts. Existing numerical examples
are illustrative calculations, not current sportsbook observations. Select `demo` explicitly
for a limited anonymous sample. Select `account` only in a private copy for requests using
your own key and allowance. The hidden key prompt appears when an account request runs.
See [current docs](https://parlay-api.com/docs) and [plans](https://parlay-api.com/pricing).


In [ ]:
# Run All is offline by default. Choose demo or account explicitly for API calls.
MODE = "offline"  # "offline", "demo", or "account"
SPORT = "baseball_mlb"
RUN_EXTRA_API_CHECKS = False
RUN_POLLING = False
SAVE_PRIVATE_CSV = False

import getpass
import json
import requests

SPORTS = {"baseball_mlb", "basketball_nba", "americanfootball_nfl",
          "icehockey_nhl", "soccer_epl", "mma_mixed_martial_arts"}
BASE_URL = "https://parlay-api.com"
_runtime_key = None

def get_runtime_key():
    global _runtime_key
    if MODE != "account":
        raise RuntimeError("Choose account mode before entering a key.")
    if _runtime_key is None:
        value = getpass.getpass("Your own ParlayAPI key (hidden; account credits apply): ")
        if not value or any(ord(c) < 33 or ord(c) > 126 for c in value):
            raise RuntimeError("Enter a valid key without whitespace or control characters.")
        _runtime_key = value
    return _runtime_key

def request_json(path, *, params=None, method="GET", body=None, account=False):
    if MODE not in {"offline", "demo", "account"} or SPORT not in SPORTS:
        raise RuntimeError("Choose a listed mode and supported sport.")
    if MODE == "offline":
        raise RuntimeError("Offline mode makes no API requests.")
    if not path.startswith("/") or path.startswith("//") or ".." in path or "\\" in path:
        raise RuntimeError("Use a fixed API path.")
    headers = {"Accept": "application/json"}
    if account:
        headers["X-API-Key"] = get_runtime_key()
    try:
        with requests.request(method, "https://parlay-api.com" + path,
                              params=params, json=body, headers=headers,
                              timeout=30, allow_redirects=False, stream=True) as response:
            if response.status_code != 200:
                raise RuntimeError(f"API returned HTTP {response.status_code}. No automatic retry.")
            chunks = []
            size = 0
            for chunk in response.iter_content(65536):
                size += len(chunk)
                if size > 10_000_000:
                    raise RuntimeError("Response exceeds the size limit.")
                chunks.append(chunk)
            return json.loads(b"".join(chunks))
    except (requests.RequestException, ValueError):
        raise RuntimeError("Request or JSON response failed. No automatic retry.") from None

if MODE not in {"offline", "demo", "account"} or SPORT not in SPORTS:
    raise RuntimeError("Choose a listed mode and supported sport.")
print("Mode:", MODE)
print("Offline runs the math without network. Demo is a limited anonymous sample.")
print("Account mode prompts at runtime and uses your own allowance. Keep that copy private.")


In [ ]:
def fetch_odds(sport=None, markets="h2h,spreads,totals", odds_format="american"):
    """A chosen demo or account request. Offline returns no live observations."""
    sport = sport or SPORT
    if sport not in SPORTS or odds_format != "american":
        raise RuntimeError("Choose a supported sport and American odds.")
    requested = markets.split(",")
    if not requested or any(m not in {"h2h", "spreads", "totals"} for m in requested):
        raise RuntimeError("Choose h2h, spreads, or totals.")
    if MODE == "offline":
        return []
    if MODE == "account":
        events = request_json(f"/v1/sports/{sport}/odds", account=True,
                              params={"markets": markets,
                                      "oddsFormat": "american"})
    else:
        payload = request_json(f"/v1/try/{sport}/odds")
        if (not isinstance(payload, dict) or payload.get("demo") is not True
                or not isinstance(payload.get("events"), list) or len(payload["events"]) > 5):
            raise RuntimeError("Unexpected demo response. No observations used.")
        events = payload["events"]
    if not isinstance(events, list) or any(not isinstance(e, dict) or e.get("sport_key") != sport for e in events):
        raise RuntimeError("Response does not match the chosen sport.")
    return events

try:
    events = fetch_odds()
except Exception:
    events = []
    print(f"Fetch failed (request failed). Check your connection or key and re-run this cell.")
print("Offline: no API request made." if MODE == "offline" else f"Returned {len(events)} {SPORT} events")
if events:
    ev = events[0]
    print("First event:", ev["away_team"], "at", ev["home_team"], "starting", ev["commence_time"])
else:
    print("No API observations loaded. Offline does not fetch data; live responses can be empty.")

## Flatten the nested JSON

One row per event, bookmaker, market and outcome. `processed_at` is the local time this
flattening function ran, not the request time or a source update. `bookmaker_last_update`
and `market_last_update` preserve the supplied source fields separately. Missing values
remain missing; none of these columns is a freshness guarantee.


In [ ]:
import pandas as pd
from datetime import datetime, timezone

def flatten(events):
    processed_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    rows = []
    for ev in events:
        for bm in ev.get("bookmakers", []):
            for mkt in bm.get("markets", []):
                for out in mkt.get("outcomes", []):
                    rows.append({
                        "processed_at": processed_at,
                        "event_id": ev.get("id"),
                        "commence_time": ev.get("commence_time"),
                        "home_team": ev.get("home_team"),
                        "away_team": ev.get("away_team"),
                        "bookmaker": bm.get("key"),
                        "bookmaker_last_update": bm.get("last_update"),
                        "market_last_update": mkt.get("last_update"),
                        "market": mkt.get("key"),
                        "outcome": out.get("name"),
                        "price": out.get("price"),
                        "point": out.get("point"),  # spread / total line, None for h2h
                    })
    return pd.DataFrame(rows)

df = flatten(events)
print(f"{len(df)} priced outcomes across {df['bookmaker'].nunique() if not df.empty else 0} bookmakers")
df.head(10)

## Best price per outcome (line shopping, with one honest guard)

American odds compare cleanly as plain numbers: a bigger number always pays more
(+120 beats -110, and -105 beats -110). So the best available price per outcome is
almost just an `idxmax`.

Almost: a thin or mismatched listing (an exchange with no liquidity, a stale
board) can post a price far better than anything a real book will honor, and a
naive max crowns exactly those rows. So first drop any price more than 15% above
the outcome's median decimal price. Production line-shopping boards apply the
same kind of outlier guard.

In [ ]:
if df.empty:
    print("Nothing to shop: no rows fetched.")
else:
    h2h = df[df["market"] == "h2h"].copy()
    # American -> decimal so prices compare on a multiplicative scale.
    h2h["decimal"] = [1 + p / 100 if p > 0 else 1 + 100 / (-p) for p in h2h["price"]]
    med = h2h.groupby(["event_id", "outcome"])["decimal"].transform("median")
    shoppable = h2h[h2h["decimal"] <= med * 1.15]
    dropped = len(h2h) - len(shoppable)
    if dropped:
        print(f"outlier guard dropped {dropped} listing(s) priced far off the market median")
    best = shoppable.loc[shoppable.groupby(["event_id", "outcome"])["price"].idxmax(),
                         ["home_team", "away_team", "outcome", "bookmaker", "price"]]
    best = best.sort_values(["home_team", "outcome"]).reset_index(drop=True)
    display(best)

## Optional private CSV export

CSV export is off by default. Enable `SAVE_PRIVATE_CSV` only in your private copy after
loading observations. The file stays in this runtime's temporary directory; do not commit
or publish it. The processing and source-time columns retain the meanings described above.


In [ ]:
if not SAVE_PRIVATE_CSV or df.empty:
    print("CSV export is off by default. Enable SAVE_PRIVATE_CSV only in your private copy.")
else:
    import tempfile
    with tempfile.NamedTemporaryFile(mode="w", prefix="parlayapi-private-", suffix=".csv", delete=False) as output_file:
        df.to_csv(output_file, index=False)
        csv_path = output_file.name
    print("Private runtime CSV created at", csv_path, ". Do not commit or publish it.")


---

**More ParlayAPI resources**

- Docs: [parlay-api.com/docs](https://parlay-api.com/docs)
- Free API key (no card): [parlay-api.com/signup](https://parlay-api.com/signup)
- Browser calculators the math here matches: [no-vig](https://parlay-api.com/tools/no-vig-calculator), [parlay](https://parlay-api.com/tools/parlay-calculator), [EV](https://parlay-api.com/tools/ev-calculator)
- The rest of this series: [github.com/JacobiusMakes/parlayapi-notebooks](https://github.com/JacobiusMakes/parlayapi-notebooks)

These notebooks are for personal and internal research and education. Nothing here is betting advice.

## Private runtime data

Keep this notebook's code shareable and your account work private. Do not paste keys into
code cells, save them in notebook text, or commit downloaded observations. The hidden prompt
keeps the key in this runtime only. Clear all outputs before sharing or saving to GitHub;
Colab's output-omission setting is an additional safeguard, not a guarantee on other hosts.
The original published notebook contains no saved API results. Do not share an executed
account notebook or its exports. Each person uses their own account and key.

The MIT license covers code. API access does not grant public redisplay or redistribution
rights. Your applicable [terms](https://parlay-api.com/terms) and written agreement govern data.
Current coverage and plans: [docs](https://parlay-api.com/docs), [pricing](https://parlay-api.com/pricing).


In [ ]:
# Drop the runtime reference when finished. Restart the runtime to release other state.
_runtime_key = None
